# Week 7 Hands-On Lab — Learning to Act, and Learning to Cheat

**ESP3201 · formative hands-on lab.** Runs on free-tier Google Colab with a **CPU runtime**. Trains in seconds.

- **Part A** — tabular **Q-learning** on Gymnasium FrozenLake.
- **Part B** — a **reward-hacking** diagnosis: a mis-specified reward makes the optimal policy farm a proxy instead of solving the task.
- **Part C** — dissect the hacked policy.
- **Part D** — **design and test your own reward fix.**

> **Report only what your run actually produced.** Learning curves are stochastic; quote your run's numbers.

**Attribution.** The Q-learning workflow mirrors the HuggingFace Deep RL Course (Unit 2) and Farama Gymnasium. The reward-hacking framing follows DeepMind's specification-gaming catalogue and OpenAI's *Faulty Reward Functions in the Wild*. All lab code is original to ESP3201.

## Setup

In [ ]:
import sys, os

COURSE_REPO_URL = ""  # PIN THIS for Colab

try:
    import numpy
except ModuleNotFoundError:
    os.system("pip install -q numpy")
try:
    import gymnasium
except ModuleNotFoundError:
    os.system("pip install -q gymnasium")
try:
    import rl_lab
except ModuleNotFoundError:
    cand = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    if os.path.exists(os.path.join(cand, "rl_lab.py")):
        sys.path.insert(0, cand)
    elif COURSE_REPO_URL:
        os.system(f"git clone -q {COURSE_REPO_URL} course_repo")
        sys.path.insert(0, "course_repo/labs/week07_reward_hacking_diagnostics/starter")
    else:
        raise ModuleNotFoundError("rl_lab.py not found. On Colab set COURSE_REPO_URL.")
    import rl_lab
from rl_lab import (q_learning, evaluate, greedy_policy, make_frozenlake,
                    RewardHackGrid, reached_goal_success)
import numpy as np
print("rl_lab loaded.")

## Part A — Q-learning on FrozenLake

In [ ]:
env = make_frozenlake(slippery=False)
ns, na = env.observation_space.n, env.action_space.n
res = q_learning(env, ns, na, episodes=3000, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_decay=0.999, max_steps=100, seed=0)
print(f"final 100-episode success rate: {res.moving_success_rate(100):.2f}")
ev = evaluate(make_frozenlake(slippery=False), res.Q)
print(f"greedy policy: return={ev['return']:.1f}  success={ev['success']}  steps={len(ev['trajectory'])-1}")

In [ ]:
import matplotlib.pyplot as plt
succ = res.successes(); window = 100
smooth = [sum(succ[max(0,i-window):i+1]) / len(succ[max(0,i-window):i+1]) for i in range(len(succ))]
plt.figure(figsize=(7,3)); plt.plot(smooth)
plt.xlabel('episode'); plt.ylabel(f'success rate ({window}-ep moving)')
plt.title('FrozenLake Q-learning'); plt.tight_layout(); plt.show()

**Try it:** sweep `alpha`, `gamma`, `epsilon_decay`. Which most changes how fast (and whether) the agent learns?

## Part B — A reward you can hack

`RewardHackGrid` is a corridor `[S][ ][P][ ][G]`. Under the **proxy** reward, stepping into the polish tile `P` pays out *every time*. Watch **reward earned** climb while **task success** stays at zero.

In [ ]:
proxy_env = RewardHackGrid(reward_mode='proxy')
hacked = q_learning(proxy_env, proxy_env.n_states, proxy_env.n_actions,
                    episodes=1500, max_steps=proxy_env.max_steps,
                    is_success=reached_goal_success, seed=0)
final_return = sum(hacked.returns()[-100:]) / 100.0
final_success = hacked.moving_success_rate(100)
print(f"MIS-SPECIFIED reward:  mean return={final_return:.1f}   goal-success rate={final_success:.2f}")
ev = evaluate(RewardHackGrid(reward_mode='proxy'), hacked.Q, is_success=reached_goal_success)
print(f"greedy trajectory (cells): {ev['trajectory']}")
print('^ it oscillates around the proxy cell and never reaches cell 4 (G).')

### The built-in fix

`reward_mode='fixed'` removes the proxy payout and adds a small step cost.

In [ ]:
fixed_env = RewardHackGrid(reward_mode='fixed', step_cost=-0.05)
fixed = q_learning(fixed_env, fixed_env.n_states, fixed_env.n_actions,
                   episodes=1500, max_steps=fixed_env.max_steps,
                   is_success=reached_goal_success, seed=0)
ev2 = evaluate(RewardHackGrid(reward_mode='fixed', step_cost=-0.05), fixed.Q, is_success=reached_goal_success)
print(f"FIXED reward:  greedy return={ev2['return']:.2f}  success={ev2['success']}  trajectory={ev2['trajectory']}")

## Part C — Dissect the hacked policy

In [ ]:
print('Greedy action per cell (0=left, 1=right):', greedy_policy(hacked.Q))
for s in range(proxy_env.n_states):
    print(f'  cell {s}: left={hacked.Q[s,0]:7.2f}  right={hacked.Q[s,1]:7.2f}')
print('\nReward spec: entering cell', proxy_env.proxy_cell, 'pays', proxy_env.proxy_reward,
      'every time; goal (cell', proxy_env.goal_cell, ') pays', proxy_env.goal_reward, 'once.')

## Part D — Design and TEST your own reward fix

Don't just propose a fix — implement it and check whether the trained policy still hacks. Write a `custom_reward(prev_pos, new_pos, entered, env)` that returns the reward for one step, then train with `reward_mode='custom'`.

A fix *works* only if the greedy policy **reaches the goal** (`success=True`). Try to make one that does — and try a tempting one that secretly still rewards the proxy and watch it fail.

In [ ]:
def my_reward(prev_pos, new_pos, entered, env):
    # EDIT THIS. Available: env.proxy_cell, env.goal_cell, env.length.
    # Example starting point (a small step cost + a goal bonus, no proxy payout):
    r = -0.05
    if new_pos == env.goal_cell:
        r += 5.0
    return r

env = RewardHackGrid(reward_mode='custom', custom_reward=my_reward)
res = q_learning(env, env.n_states, env.n_actions, episodes=1500,
                 max_steps=env.max_steps, is_success=reached_goal_success, seed=0)
ev = evaluate(RewardHackGrid(reward_mode='custom', custom_reward=my_reward), res.Q,
              is_success=reached_goal_success)
print(f"YOUR reward:  reached_goal={ev['success']}  return={ev['return']:.2f}  trajectory={ev['trajectory']}")
print('A working fix makes reached_goal=True.')

## Worksheet (your deliverable)

### 1. Reward-vs-success table

| Reward design | Mean return | Goal-success rate |
|---------------|-------------|-------------------|
| proxy (mis-specified) | | |
| fixed (built-in) | | |
| **your custom fix** | | |

### 2. Diagnose the hack

- Which cell/action does the hacked greedy policy exploit, and what in the **reward spec** makes it profitable? Cite the Q-values.
- Why is *mean return* a misleading success metric here?

### 3. Your fix

- Paste your `custom_reward`. Did it make `reached_goal=True`?
- If your first attempt failed, what did the policy do instead, and what did you change?
- `What you re-measured to confirm the fix (not just higher reward):`

### 4. Connect to deployment

- In one sentence: where might a real robot reward function hide a proxy like this one?

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking